In [15]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from pathlib import Path

from stock_mpt import StockMPT, LinearModel, NaiveModel
from dataloader_builder_mpt import build_dataloaders
from setup import StockMPT_cfg, LinearModel_cfg, NaiveModel_cfg, SEED
from setup import path_data_preprocessor
from model_training_mpt import model_setup, train_model_cuda

from model_training_mpt import train_model_cuda, evaluate_model, evaluate_best_model, precision_recall_curve
from model_analysis_mpt import test_model, print_loss_analysis, process_losses, format_num

from dataset_analysis import analyze_ticker_days, compare_outliers, feature_correlations, regime_analysis

In [16]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [17]:
torch.manual_seed(SEED)
print(path_data_preprocessor)
dls, train_norms = build_dataloaders(path_data_preprocessor)

counts = torch.zeros(3, dtype=torch.long)

for _, y in dls["train"]:
    counts += torch.bincount(y.flatten().cpu(), minlength=3)

print("Counts:", counts)
print("Distribution:", counts / counts.sum())

preprocessed_data/data_1min_2021_2026_4
Building DataLoaders...
Train dataset samples: 32,361
Train loader batches:  252
Batch size:            128
Counts: tensor([3500113, 5328931, 3589516])
Distribution: tensor([0.2818, 0.4291, 0.2890])


In [18]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 10

eval_bs = 1000

stockMPT, stockMPT_params, opt1, sca1, sch1 = model_setup(StockMPT, StockMPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

3481600
4864


NaiveModel()

In [19]:
model_train_losses, model_val_losses = train_model_cuda(stockMPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 1:

Learning Rate: 4.00e-04



KeyboardInterrupt: 

## Model Analysis -------------------------

In [ ]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
mpt_losses = evaluate_best_model(stockMPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
mpt_test_losses = test_model(dls["test"], stockMPT, device, eval_bs, analysis_pbar)


|███▎      | 33.0% (00:05) Evaluating model on training data... (12/252) [346/1050]:                      

[] []


|██████▍   | 64.4% (00:10) Evaluating model on training data... (9/252) [676/1050]:   

[] []


|██████████| 100.0% (00:48) Evaluating model on testing data... (16/17) [1050/1050]:  

In [ ]:
for key, features in [("CE", StockMPT_cfg["target_features"]),
                      ("ACC", StockMPT_cfg["target_features"]),
                      ("PREC", StockMPT_cfg["target_features"]),
                      ("REC", StockMPT_cfg["target_features"]),
                      ("F1", StockMPT_cfg["target_features"])]:
    print_loss_analysis(
        process_losses(mpt_losses + mpt_test_losses +
                       linear_losses + linear_test_losses +
                       naive_losses + naive_test_losses, key),
        [stockMPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
        [format_num(stockMPT_params), format_num(linearModel_params), "0"],
        features, key
    )


--------------------------------------------------------------------------------------------------------------

CE

--------------------------------------------------------------------------------------------------------------

StockMPT-v15-4-5-183-s3: 3.5M
    Training:       0.9236
    Validation:     0.9407
    Testing:        0.9209
    
LinearModel-v15-111-4-183-30: 4.9K
    Training:       1.0088
    Validation:     1.0201
    Testing:        1.0066
    
NaiveModel-B1_183: 0
    Training:       1.0872
    Validation:     1.1036
    Testing:        1.0786
    

--------------------------------------------------------------------------------------------------------------



--------------------------------------------------------------------------------------------------------------

ACC

--------------------------------------------------------------------------------------------------------------

StockMPT-v15-4-5-183-s3: 3.5M
    Training:       0.5493
    Validation:     0.5330

In [ ]:
precision_recall_curve(dls["val"], stockMPT, device, cls=2)
print("--------------")
precision_recall_curve(dls["val"], stockMPT, device, cls=1)
print("--------------")
precision_recall_curve(dls["val"], stockMPT, device, cls=0)

0.10 | PREC 0.3204 | REC 0.9811 | N 3668063
0.15 | PREC 0.3394 | REC 0.9482 | N 3347374
0.20 | PREC 0.3588 | REC 0.8988 | N 3000910
0.25 | PREC 0.3791 | REC 0.8287 | N 2618752
0.30 | PREC 0.4014 | REC 0.7293 | N 2176436
0.35 | PREC 0.4266 | REC 0.5876 | N 1650098
0.40 | PREC 0.4586 | REC 0.3953 | N 1032562
0.45 | PREC 0.5032 | REC 0.1891 | N 450139
0.50 | PREC 0.5567 | REC 0.0747 | N 160702
0.55 | PREC 0.6044 | REC 0.0324 | N 64173
0.60 | PREC 0.6572 | REC 0.0149 | N 27171
0.65 | PREC 0.7061 | REC 0.0065 | N 11037
0.70 | PREC 0.7566 | REC 0.0024 | N 3735
0.75 | PREC 0.8043 | REC 0.0006 | N 884
0.80 | PREC 0.8409 | REC 0.0001 | N 88
0.85 | PREC 1.0000 | REC 0.0000 | N 2
0.90 | PREC 0.0000 | REC 0.0000 | N 0
0.95 | PREC 0.0000 | REC 0.0000 | N 0
--------------


|██████████| 100.0% (01:01) Evaluating model on testing data... (16/17) [1050/1050]: 

0.10 | PREC 0.4404 | REC 0.9848 | N 3616018
0.15 | PREC 0.4751 | REC 0.9568 | N 3256647
0.20 | PREC 0.5128 | REC 0.9156 | N 2887595
0.25 | PREC 0.5491 | REC 0.8656 | N 2549383
0.30 | PREC 0.5836 | REC 0.8106 | N 2246038
0.35 | PREC 0.6163 | REC 0.7525 | N 1974585
0.40 | PREC 0.6461 | REC 0.6929 | N 1734456
0.45 | PREC 0.6743 | REC 0.6315 | N 1514338
0.50 | PREC 0.7018 | REC 0.5674 | N 1307643
0.55 | PREC 0.7283 | REC 0.5015 | N 1113573
0.60 | PREC 0.7545 | REC 0.4337 | N 929657
0.65 | PREC 0.7802 | REC 0.3629 | N 752174
0.70 | PREC 0.8059 | REC 0.2884 | N 578785
0.75 | PREC 0.8329 | REC 0.2134 | N 414329
0.80 | PREC 0.8611 | REC 0.1415 | N 265665
0.85 | PREC 0.8883 | REC 0.0824 | N 150060
0.90 | PREC 0.9145 | REC 0.0400 | N 70712
0.95 | PREC 0.9525 | REC 0.0067 | N 11361
--------------
0.10 | PREC 0.3197 | REC 0.9779 | N 3599177
0.15 | PREC 0.3394 | REC 0.9447 | N 3274651
0.20 | PREC 0.3601 | REC 0.8971 | N 2931431
0.25 | PREC 0.3817 | REC 0.8314 | N 2562806
0.30 | PREC 0.4048 | REC 0.

In [ ]:
handpicked_dls, train_norms = build_dataloaders("handpicked_data", False, drop_last = False)

Building DataLoaders...


In [ ]:
test_losses = test_model(handpicked_dls["test"], stockMPT, device, eval_bs)
print(test_losses)
precision_recall_curve(handpicked_dls["test"], stockMPT, device, cls=2)
print("-")
precision_recall_curve(handpicked_dls["test"], stockMPT, device, cls=1)
print("-")
precision_recall_curve(handpicked_dls["test"], stockMPT, device, cls=0)

({'LOSS': tensor(1.9740, device='cuda:0'), 'CE': tensor(1.2802, device='cuda:0'), 'PREC_LOSS': tensor(0.6937, device='cuda:0'), 'ACC': tensor(0.2935, device='cuda:0'), 'PREC': tensor([0.3022, 0.1675, 0.4639], device='cuda:0'), 'REC': tensor([0.7198, 0.0732, 0.1271], device='cuda:0'), 'F1': tensor([0.4257, 0.1019, 0.1996], device='cuda:0')},)
0.10 | PREC 0.3065 | REC 1.0000 | N 1155
0.15 | PREC 0.3065 | REC 1.0000 | N 1155
0.20 | PREC 0.3065 | REC 1.0000 | N 1155
0.25 | PREC 0.3065 | REC 1.0000 | N 1155
0.30 | PREC 0.3045 | REC 0.9859 | N 1146
0.35 | PREC 0.2582 | REC 0.6412 | N 879
0.40 | PREC 0.3265 | REC 0.0452 | N 49
0.45 | PREC 0.2727 | REC 0.0169 | N 22
0.50 | PREC 0.0000 | REC 0.0000 | N 7
0.55 | PREC 0.0000 | REC 0.0000 | N 2
0.60 | PREC 0.0000 | REC 0.0000 | N 0
0.65 | PREC 0.0000 | REC 0.0000 | N 0
0.70 | PREC 0.0000 | REC 0.0000 | N 0
0.75 | PREC 0.0000 | REC 0.0000 | N 0
0.80 | PREC 0.0000 | REC 0.0000 | N 0
0.85 | PREC 0.0000 | REC 0.0000 | N 0
0.90 | PREC 0.0000 | REC 0.00

In [ ]:
df = analyze_ticker_days(
    model=stockMPT,
    dataloader=dls["val"],
    device=device,
    batch_size=128,
    output_path=Path("dataset_analysis/val_ticker_days.parquet"),
)
print(df)

Analyzed 10,368/10,487 ticker-days
Saved: dataset_analysis\val_ticker_days.parquet
Ticker-days: 10,368
Mean CE: 0.9407
Median CE: 0.9898
Mean ACC: 0.5330
         Tk        date  PRICE_OPEN  PRICE_CLOSE  PRICE_MEAN  PRICE_MIN  \
0      SGRP  2025-08-26       1.180        1.280    1.208767     1.1200   
1      SGRP  2025-08-27       1.330        1.310    1.350441     1.3000   
2      SGRP  2025-08-28       1.300        1.171    1.215796     1.1601   
3      RAVE  2025-09-12       3.230        3.550    3.426335     3.2300   
4      ELDN  2025-09-26       2.570        2.685    2.603744     2.4400   
...     ...         ...         ...          ...         ...        ...   
10363  REED  2025-12-11       3.070        3.540    3.231074     3.0700   
10364  REED  2025-12-12       3.410        3.620    3.604645     3.3300   
10365  REED  2026-01-07       2.125        2.300    2.369756     2.1250   
10366   AZI  2026-02-10      11.006        7.450    7.724312     6.5000   
10367   AZI  2026-03-

In [ ]:
best, worst, comparison = compare_outliers(df, tail=0.10)
corr = feature_correlations(df)

print(comparison.head(25))
print(corr.head(25))

                    BEST_MEDIAN  WORST_MEDIAN    ALL_MEDIAN  WORST_MINUS_BEST  \
TOTAL_VOLUME      392054.625000  2.315306e+06  1.363304e+06      1.923251e+06   
RTH_VOLUME        383760.000000  2.243473e+06  1.318390e+06      1.859713e+06   
MAX_VOLUME         52785.000000  1.302460e+05  9.004900e+04      7.746100e+04   
PM_VOLUME              0.000000  7.421000e+03  5.491500e+03      7.421000e+03   
MEAN_VOLUME          933.463379  5.512633e+03  3.245961e+03      4.579170e+03   
DOWN40_N               2.000000  8.200000e+01  7.000000e+01      8.000000e+01   
UP40_N                 3.000000  7.100000e+01  6.200000e+01      6.800000e+01   
MAX_RV                61.639370  2.845616e+01  3.551051e+01     -3.318321e+01   
DOWN40_TP              0.000000  3.300000e+01  3.200000e+01      3.300000e+01   
UP40_TP                1.000000  2.900000e+01  2.700000e+01      2.800000e+01   
DOWN45_N               0.000000  2.700000e+01  3.000000e+01      2.700000e+01   
UP45_N                 0.000

In [ ]:
import importlib
import dataset_analysis

importlib.reload(dataset_analysis)
regime_analysis = dataset_analysis.regime_analysis

for feature in [
    "RANGE_PCT",
    "TOTAL_VOLUME",
    "FILL_RATE",
    "MEAN_ABS_MACD",
    "MAX_ABS_GP",
]:
    print(f"\n{'='*80}")
    print(feature)
    print(regime_analysis(df, feature, show_recall = False, show_n = True).to_string(index=False))


RANGE_PCT
            BIN    N  FEATURE_MEDIAN       CE      ACC  UP40_PREC  UP40_N  DOWN40_PREC  DOWN40_N  UP45_PREC  UP45_N  DOWN45_PREC  DOWN45_N  UP50_PREC  UP50_N  DOWN50_PREC  DOWN50_N  UP55_PREC  UP55_N  DOWN55_PREC  DOWN55_N  UP60_PREC  UP60_N  DOWN60_PREC  DOWN60_N  UP65_PREC  UP65_N  DOWN65_PREC  DOWN65_N  UP70_PREC  UP70_N  DOWN70_PREC  DOWN70_N
 (0.049, 0.123] 2074        0.089655 0.883083 0.584909   0.425077  107691     0.441932    102398   0.466420   41156     0.483677     40097   0.519878   13432     0.527502     14508   0.588094    4569     0.572492      5373   0.664533    1562     0.642019      1961   0.676026     463     0.737132       544   0.790476     105     0.793651       126
 (0.123, 0.188] 2073        0.155080 0.934991 0.543628   0.447058  165437     0.452986    167067   0.491756   69142     0.498355     69888   0.554406   22608     0.545139     25322   0.604405    8491     0.598972      8947   0.662018    3370     0.655838      3417   0.721591    1232     0.7